In [1]:
from pathlib import Path
import pandas as pd

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_seq_items', None)

In [3]:
BASE_DIR = Path(r"C:\Users\Kevin Strey\Desktop\Feature-models")
HISTORY_DIR = BASE_DIR / "2-JMethodsExtractor" / "target" / "results"
UNDERSTAND_DIR = BASE_DIR / "2.understand" / "results"
CK_DIR = BASE_DIR / "4-CKHistoryExtractor" / "target" / "results"
IGNORED_METHODS_LOG = BASE_DIR / "7-DadosML" / "logs" / "merge_v2_ignored_methods.csv"

def find_csvs(folder, predicate):
    paths = sorted(path for path in folder.glob("*.csv") if predicate(path))
    if not paths:
        raise FileNotFoundError(f"Nenhum CSV encontrado em {folder}")
    return paths

ck_files = find_csvs(
    CK_DIR, lambda path: path.name.endswith("_method_history.csv")
)
history_files = find_csvs(
    HISTORY_DIR,
    lambda path: path.name.endswith("_history.csv") and not any(
        suffix in path.stem
        for suffix in ("_method_history", "_class_history", "_understand_history")
    ),
)
understand_files = find_csvs(
    UNDERSTAND_DIR, lambda path: path.name.endswith("_understand_history.csv")
)

print("Pastas utilizadas:")
print("CK:", CK_DIR)
print("Histórico:", HISTORY_DIR)
print("Understand:", UNDERSTAND_DIR)
print("Arquivos encontrados:", len(ck_files), len(history_files), len(understand_files))


Pastas utilizadas:
CK: C:\Users\Kevin Strey\Desktop\Feature-models\4-CKHistoryExtractor\target\results
Histórico: C:\Users\Kevin Strey\Desktop\Feature-models\2-JMethodsExtractor\target\results
Understand: C:\Users\Kevin Strey\Desktop\Feature-models\2.understand\results
Arquivos encontrados: 19 19 19


In [4]:
chaves = ["project", "release", "commitHash", "methodId"]
ignored_methods = []

def remove_conflicting_keys(df, name):
    duplicate_rows = df[df.duplicated(chaves, keep=False)]
    if duplicate_rows.empty:
        return df.drop_duplicates(chaves, keep="first")
    value_columns = [column for column in df.columns if column not in chaves]
    conflicts = (
        duplicate_rows.groupby(chaves, dropna=False)[value_columns]
        .nunique(dropna=False).gt(1).any(axis=1)
    )
    conflicting_keys = conflicts[conflicts].index
    if len(conflicting_keys):
        duplicate_index = df.set_index(chaves).index
        df = df.loc[~duplicate_index.isin(conflicting_keys)]
        print(f"{name}: {len(conflicting_keys)} chaves conflitantes excluídas")
    return df.drop_duplicates(chaves, keep="first")

def merge_project(ck_path, history_path, understand_path):
    ck = pd.read_csv(ck_path)
    history = pd.read_csv(history_path)
    understand = pd.read_csv(understand_path)

    ck["loc"] = pd.to_numeric(ck["loc"], errors="coerce")
    understand["CountLine"] = pd.to_numeric(
        understand["CountLine"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    history["LOC"] = pd.to_numeric(
        history["LOC"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    ck = ck[ck["loc"] > 0]
    understand = understand[understand["CountLine"] > 0]
    history = history.sort_values("commit_index").drop_duplicates(
        ["project", "release", "methodId"], keep="last"
    )
    history = history[history["LOC"] > 0]

    snapshots = pd.concat(
        [ck[["project", "release", "commitHash"]],
         understand[["project", "release", "commitHash"]]],
        ignore_index=True,
    ).drop_duplicates()
    counts = snapshots.groupby(["project", "release"]).size()
    if (counts > 1).any():
        raise ValueError(f"Hashes incompatíveis em {ck_path.stem}: {counts[counts > 1].index.tolist()}")
    history = history.drop(columns=["commitHash"]).merge(
        snapshots, on=["project", "release"], how="inner", validate="many_to_one"
    )
    for frame, name in [(ck, "CK"), (history, "Histórico"), (understand, "Understand")]:
        valid_method_ids = frame["methodId"].astype(str).str.count("::").ge(2)
        invalid = frame.loc[~valid_method_ids, ["project", "release", "methodId"]].copy()
        if not invalid.empty:
            invalid["dataset"] = name
            invalid["reason"] = "methodId sem arquivo::classe::metodo"
            ignored_methods.extend(invalid.to_dict("records"))
            print(f"{name}: {len(invalid)} methodId inválidos ignorados")
        frame = frame.loc[valid_method_ids].copy()
        frame = remove_conflicting_keys(frame, name)
        if name == "CK": ck = frame
        elif name == "Histórico": history = frame
        else: understand = frame
    return ck.merge(history, on=chaves, how="inner", validate="one_to_one", suffixes=("_ck", "_history")).merge(
        understand, on=chaves, how="inner", validate="one_to_one", suffixes=("", "_understand")
    )

ck_by_project = {path.stem.removesuffix("_method_history"): path for path in ck_files}
history_by_project = {path.stem.removesuffix("_history"): path for path in history_files}
understand_by_project = {path.stem.removesuffix("_understand_history"): path for path in understand_files}
common_projects = sorted(set(ck_by_project) & set(history_by_project) & set(understand_by_project))
if not common_projects:
    raise ValueError("Nenhum projeto possui os três datasets")
parts = []
for project in common_projects:
    part = merge_project(ck_by_project[project], history_by_project[project], understand_by_project[project])
    print(project, "->", len(part), "linhas")
    parts.append(part)
df_final = pd.concat(parts, ignore_index=True)
assert not df_final.duplicated(chaves).any()
IGNORED_METHODS_LOG.parent.mkdir(parents=True, exist_ok=True)
ignored_log = pd.DataFrame(
    ignored_methods,
    columns=["dataset", "project", "release", "methodId", "reason"],
)
ignored_log.to_csv(IGNORED_METHODS_LOG, index=False, encoding="utf-8-sig")
print("Métodos ignorados:", len(ignored_log))
print("Log salvo em:", IGNORED_METHODS_LOG)
if not ignored_log.empty:
    print(ignored_log.groupby(["dataset", "project"]).size().rename("count"))

CK: 44 chaves conflitantes excluídas
BIRT -> 222946 linhas
CK: 55 chaves conflitantes excluídas
Understand: 561 chaves conflitantes excluídas
CDT -> 179247 linhas
Understand: 365 chaves conflitantes excluídas
CVS -> 25611 linhas
DataTools -> 56557 linhas
Understand: 688 chaves conflitantes excluídas
EGit -> 17345 linhas
CK: 110 chaves conflitantes excluídas
Histórico: 3 methodId inválidos ignorados
Understand: 129 chaves conflitantes excluídas
EMF -> 56565 linhas
CK: 66 chaves conflitantes excluídas
Histórico: 10 methodId inválidos ignorados
Understand: 1 chaves conflitantes excluídas
EclipseLink -> 667318 linhas
CK: 330 chaves conflitantes excluídas
Histórico: 23 methodId inválidos ignorados
Understand: 26 chaves conflitantes excluídas
GEF -> 23073 linhas
CK: 92 chaves conflitantes excluídas
GMF Runtime -> 12659 linhas
CK: 464 chaves conflitantes excluídas
Understand: 344 chaves conflitantes excluídas
JDT -> 32300 linhas
CK: 16 chaves conflitantes excluídas
Understand: 153 chaves conf

In [13]:
df_final.sample(20)

,project,release,commitIndex,commitHash,methodId,constructor,line,cbo,cboModified,fanin,fanout,wmc,rfc,loc,returnsQty,variablesQty,parametersQty,methodsInvokedQty,methodsInvokedLocalQty,methodsInvokedIndirectLocalQty,loopQty,comparisonsQty,tryCatchQty,parenthesizedExpsQty,stringLiteralsQty,numbersQty,assignmentsQty,mathOperationsQty,maxNestedBlocksQty,anonymousClassesQty,innerClassesQty,lambdasQty,uniqueWordsQty,modifiers,logStatementsQty,hasJavaDoc,commit_index,BOM,TACH,CHD,FCH,LCH,FRCH,WCH,WCD,CSB,CSBS,ACDF,LOC,commitIndex_understand,AvgCyclomatic,AvgCyclomaticModified,AvgCyclomaticStrict,AvgEssential,AvgLine,AvgLineBlank,AvgLineCode,AvgLineComment,CountClassBase,CountClassCoupled,CountClassDerived,CountDeclClass,CountDeclClassMethod,CountDeclClassVariable,CountDeclFile,CountDeclFunction,CountDeclInstanceMethod,CountDeclInstanceVariable,CountDeclMethod,CountDeclMethodAll,CountDeclMethodDefault,CountDeclMethodPrivate,CountDeclMethodProtected,CountDeclMethodPublic,CountInput,CountLine,CountLineBlank,CountLineCode,CountLineCodeDecl,CountLineCodeExe,CountLineComment,CountOutput,CountPath,CountSemicolon,CountStmt,CountStmtDecl,CountStmtExe,Cyclomatic,CyclomaticModified,CyclomaticStrict,Essential,MaxCyclomatic,MaxCyclomaticModified,MaxCyclomaticStrict,MaxEssential,MaxInheritanceTree,MaxNesting,PercentLackOfCohesion,RatioCommentToCode,SumCyclomatic,SumCyclomaticModified,SumCyclomaticStrict,SumEssential,loc_ck_num,CountLine_num
1669174,Scout,2024-09,53,23fb4663626c3f6b62f6a5a7023850bf8d415d3a,org.eclipse.scout.rt.ui.html/src/main/java/org/eclipse/scout/rt/ui/html/json/desktop/hybrid/JsonHybridManager.java::org.eclipse.scout.rt.ui.html.json.desktop.hybrid.JsonHybridManager::initJsonProperties(T),False,68,4,2,0,2,1,3,33,0,0,1,3,0,0,0,0,0,0,0,0,0,0,0,1,0,0,36,4,0,False,19225,18408,14,"0,3500",18997,19225,2,"14,0000","0,3500",28,"0,7000","0,3150",40,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,45.0,4.0,40.0,3.0,29.0,1.0,3.0,1.0,13.0,22.0,6.0,17.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,"0,03",1,1,1,1,33,45.0
998742,EclipseLink,2019-12,34,a5b5fb08eb1124d073fc37b4f7c5c315fdeac89a,foundation/org.eclipse.persistence.core/src/main/java/org/eclipse/persistence/oxm/mappings/XMLAnyObjectMapping.java::org.eclipse.persistence.oxm.mappings.XMLAnyObjectMapping::setMixedContent(boolean),False,668,0,0,0,0,1,0,3,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,4,1,0,False,11783,11783,0,"0,0000",0,0,0,"0,0000","0,0000",0,"0,0000","0,0000",4,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,4.0,0.0,4.0,2.0,1.0,0.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,"0,00",1,1,1,1,3,4.0
1197843,EclipseLink,2023-03,47,8ac1935575d461d3a51672c92a16062645e11bb3,jpa/eclipselink.jpa.testapps/jpa.test.xml.composite.advanced/src/test/java/org/eclipse/persistence/testing/tests/jpa/xml/composite/advanced/EmployeePopulator.java::org.eclipse.persistence.testing.tests.jpa.xml.composite.advanced.EmployeePopulator::addressExample11(),False,135,1,7,1,6,1,5,9,1,1,0,5,0,0,0,0,0,0,5,0,1,0,0,0,0,0,4,1,0,False,12758,12758,0,"0,0000",0,0,0,"0,0000","0,0000",0,"0,0000","0,0000",9,47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.0,1.0,9.0,2.0,7.0,0.0,7.0,1.0,7.0,8.0,2.0,7.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,"0,00",1,1,1,1,9,10.0
1763075,Window Builder,2023-09,37,5749a0877030439f5746261d072a120ed8ef4b78,org.eclipse.wb.tests/src/org/eclipse/wb/tests/designer/swing/model/layout/spring/SpringLayoutGefTest.java::org.eclipse.wb.tests.designer.swing.model.layout.spring.SpringLayoutGefTest::test_CREATE_absoluteSnap_rightOffset(),False,177,3,2,0,2,1,8,5,0,0,0,8,1,2,0,0,0,0,12,2,0,0,0,0,0,0,27,1,0,False,1059,1,3,"0,1667",1036,1059,2,"3,0000","0,1667",35,"2,0588","1,0245",18,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,18.0,0.0,18.0,2.0,15.0,0.0,8.0,

In [6]:
assert not df_final.duplicated(chaves).any(), (
    "Existem chaves duplicadas no df_final"
)

assert df_final[chaves].drop_duplicates().shape[0] == len(df_final)

print("OK: cada linha representa uma única combinação de método e release")

OK: cada linha representa uma única combinação de método e release


In [7]:
print("Linhas finais:", len(df_final))
print("Métodos únicos:", df_final["methodId"].nunique())
print("Chaves únicas:", df_final[chaves].drop_duplicates().shape[0])

Linhas finais: 1797604
Métodos únicos: 1022300
Chaves únicas: 1797604


In [8]:
assert df_final["project"].notna().all()
assert df_final["release"].notna().all()
assert df_final["commitHash"].notna().all()
assert df_final["commitHash"].astype(str).str.len().gt(0).all()

print("OK: projeto, release e commitHash são consistentes")
print("Projetos:", sorted(df_final["project"].unique()))
print("Releases:", df_final["release"].nunique())
print("Hashes:", df_final["commitHash"].nunique())

OK: projeto, release e commitHash são consistentes
Projetos: ['BIRT', 'CDT', 'CVS', 'DataTools', 'EGit', 'EMF', 'EclipseLink', 'GEF', 'GMF Runtime', 'JDT', 'Mylyn', 'PDE', 'PTP', 'RAP', 'Scout', 'Subversive', 'WebTools', 'Window Builder', 'm2e']
Releases: 59
Hashes: 717


In [9]:
assert (df_final["LOC"] > 0).all(), (
    "Existem métodos com LOC menor ou igual a zero"
)

assert df_final["LOC"].notna().all(), (
    "Existem valores nulos em LOC"
)

print("OK: todos os métodos têm LOC positivo")

OK: todos os métodos têm LOC positivo


In [10]:
df_final["loc_ck_num"] = pd.to_numeric(
    df_final["loc"],
    errors="coerce"
)

df_final["CountLine_num"] = pd.to_numeric(
    df_final["CountLine"],
    errors="coerce"
)

assert (df_final["loc_ck_num"] > 0).all(), (
    "CK possui métodos com LOC menor ou igual a zero"
)

assert (
    df_final["CountLine_num"].dropna() > 0
).all(), (
    "Understand possui CountLine menor ou igual a zero"
)

print("OK: CK e Understand indicam métodos ativos")

OK: CK e Understand indicam métodos ativos


In [11]:
print("Valores nulos por coluna:")
nulos = df_final.isna().sum()
print(nulos[nulos > 0].sort_values(ascending=False))

Valores nulos por coluna:
AvgCyclomatic                1797604
AvgCyclomaticModified        1797604
AvgCyclomaticStrict          1797604
AvgEssential                 1797604
AvgLine                      1797604
AvgLineBlank                 1797604
AvgLineCode                  1797604
AvgLineComment               1797604
CountClassBase               1797604
CountClassCoupled            1797604
CountClassDerived            1797604
CountDeclClass               1797604
CountDeclClassMethod         1797604
CountDeclClassVariable       1797604
CountDeclFile                1797604
CountDeclFunction            1797604
CountDeclInstanceMethod      1797604
CountDeclInstanceVariable    1797604
CountDeclMethod              1797604
CountDeclMethodAll           1797604
CountDeclMethodDefault       1797604
CountDeclMethodPrivate       1797604
CountDeclMethodProtected     1797604
CountDeclMethodPublic        1797604
MaxCyclomatic                1797604
MaxCyclomaticModified        1797604
MaxCyclomati

In [12]:
percentual_nulo = (
    df_final.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(percentual_nulo[percentual_nulo > 0])

CountDeclFile                100.0
CountDeclFunction            100.0
AvgCyclomatic                100.0
AvgCyclomaticStrict          100.0
AvgCyclomaticModified        100.0
AvgLineBlank                 100.0
AvgLine                      100.0
AvgLineCode                  100.0
AvgEssential                 100.0
AvgLineComment               100.0
CountClassBase               100.0
PercentLackOfCohesion        100.0
MaxEssential                 100.0
MaxCyclomaticStrict          100.0
MaxCyclomaticModified        100.0
MaxCyclomatic                100.0
MaxInheritanceTree           100.0
CountDeclMethodPublic        100.0
CountDeclMethodProtected     100.0
CountDeclMethodDefault       100.0
CountDeclMethodPrivate       100.0
CountDeclMethod              100.0
CountDeclInstanceVariable    100.0
CountDeclInstanceMethod      100.0
CountDeclMethodAll           100.0
CountDeclClass               100.0
CountDeclClassMethod         100.0
CountClassCoupled            100.0
CountClassDerived   

In [16]:
# Identifica as colunas que possuem pelo menos um valor NaN
colunas_nan = df_final.columns[df_final.isna().any()].tolist()

print("Colunas removidas:")
for coluna in colunas_nan:
    quantidade_nan = df_final[coluna].isna().sum()
    print(f"- {coluna}: {quantidade_nan} valores NaN")

# Atualiza o df_final removendo as colunas com NaN
df_final = df_final.drop(columns=colunas_nan)

print("\nDimensão atualizada do df_final:")
print(f"Linhas: {df_final.shape[0]}")
print(f"Colunas: {df_final.shape[1]}")

# Confirma que não restaram valores NaN
assert not df_final.isna().any().any(), (
    "Ainda existem valores NaN no df_final"
)

print("OK: nenhuma coluna com NaN permanece no df_final")

Colunas removidas:
- AvgCyclomatic: 1797604 valores NaN
- AvgCyclomaticModified: 1797604 valores NaN
- AvgCyclomaticStrict: 1797604 valores NaN
- AvgEssential: 1797604 valores NaN
- AvgLine: 1797604 valores NaN
- AvgLineBlank: 1797604 valores NaN
- AvgLineCode: 1797604 valores NaN
- AvgLineComment: 1797604 valores NaN
- CountClassBase: 1797604 valores NaN
- CountClassCoupled: 1797604 valores NaN
- CountClassDerived: 1797604 valores NaN
- CountDeclClass: 1797604 valores NaN
- CountDeclClassMethod: 1797604 valores NaN
- CountDeclClassVariable: 1797604 valores NaN
- CountDeclFile: 1797604 valores NaN
- CountDeclFunction: 1797604 valores NaN
- CountDeclInstanceMethod: 1797604 valores NaN
- CountDeclInstanceVariable: 1797604 valores NaN
- CountDeclMethod: 1797604 valores NaN
- CountDeclMethodAll: 1797604 valores NaN
- CountDeclMethodDefault: 1797604 valores NaN
- CountDeclMethodPrivate: 1797604 valores NaN
- CountDeclMethodProtected: 1797604 valores NaN
- CountDeclMethodPublic: 1797604 valo

In [17]:
df_final

,project,release,commitIndex,commitHash,methodId,constructor,line,cbo,cboModified,fanin,fanout,wmc,rfc,loc,returnsQty,variablesQty,parametersQty,methodsInvokedQty,methodsInvokedLocalQty,methodsInvokedIndirectLocalQty,loopQty,comparisonsQty,tryCatchQty,parenthesizedExpsQty,stringLiteralsQty,numbersQty,assignmentsQty,mathOperationsQty,maxNestedBlocksQty,anonymousClassesQty,innerClassesQty,lambdasQty,uniqueWordsQty,modifiers,logStatementsQty,hasJavaDoc,commit_index,BOM,TACH,CHD,FCH,LCH,FRCH,WCH,WCD,CSB,CSBS,ACDF,LOC,commitIndex_understand,CountInput,CountLine,CountLineBlank,CountLineCode,CountLineCodeDecl,CountLineCodeExe,CountLineComment,CountOutput,CountPath,CountSemicolon,CountStmt,CountStmtDecl,CountStmtExe,Cyclomatic,CyclomaticModified,CyclomaticStrict,Essential,MaxNesting,RatioCommentToCode,SumCyclomatic,SumCyclomaticModified,SumCyclomaticStrict,SumEssential,loc_ck_num,CountLine_num
0,BIRT,JunoSR0,1,3767e5c39c0eb69dfb7b789045156993d93aaf03,data/org.eclipse.birt.data.tests/test/org/eclipse/birt/data/engine/olap/data/util/PrimarySortedStackTest.java::org.eclipse.birt.data.engine.olap.data.util.PrimarySortedStackTest::testNotDistinctAsc(),False,110,1,1,0,1,6,5,34,0,4,0,5,0,0,2,2,1,0,0,20,6,0,3,0,0,0,19,1,0,False,8857,8857,0,"0,0000",0,0,0,"0,0000","0,0000",0,"0,0000","0,0000",43,1,1.0,45.0,1.0,43.0,4.0,28.0,1.0,7.0,10.0,24.0,30.0,5.0,26.0,6.0,6.0,6.0,1.0,3.0,"0,02",6,6,6,1,34,45.0
1,BIRT,JunoSR0,1,3767e5c39c0eb69dfb7b789045156993d93aaf03,data/org.eclipse.birt.data.tests/test/org/eclipse/birt/data/engine/olap/data/util/PrimarySortedStackTest.java::org.eclipse.birt.data.engine.olap.data.util.PrimarySortedStackTest::setUp(),False,33,0,0,0,0,1,1,3,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,4,0,False,8857,8857,0,"0,0000",0,0,0,"0,0000","0,0000",0,"0,0000","0,0000",4,1,0.0,4.0,0.0,4.0,1.0,1.0,0.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,"0,00",1,1,1,1,3,4.0
2,BIRT,JunoSR0,1,3767e5c39c0eb69dfb7b789045156993d93aaf03,data/org.eclipse.birt.data.tests/test/org/eclipse/birt/data/engine/olap/data/util/PrimarySortedStackTest.java::org.eclipse.birt.data.engine.olap.data.util.PrimarySortedStackTest::testDistinctAsc(),False,46,1,1,0,1,4,5,22,0,4,0,5,0,0,2,0,1,0,0,10,4,0,2,0,0,0,18,1,0,False,8857,8857,0,"0,0000",0,0,0,"0,0000","0,0000",0,"0,0000","0,0000",29,1,1.0,31.0,1.0,29.0,4.0,18.0,1.0,7.0,5.0,16.0,20.0,5.0,16.0,4.0,4.0,4.0,1.0,2.0,"0,03",4,4,4,1,22,31.0
3,BIRT,JunoSR0,1,3767e5c39c0eb69dfb7b789045156993d93aaf03,data/org.eclipse.birt.data.tests/test/org/eclipse/birt/data/engine/olap/data/util/PrimarySortedStackTest.java::org.eclipse.birt.data.engine.olap.data.util.PrimarySortedStackTest::tearDown(),False,41,0,0,0,0,1,1,3,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,4,0,False,8857,8857,0,"0,0000",0,0,0,"0,0000","0,0000",0,"0,0000","0,0000",4,1,0.0,4.0,0.0,4.0,1.0,1.0,0.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,"0,00",1,1,1,1,3,4.0
4,BIRT,JunoSR0,1,3767e5c39c0eb69dfb7b789045156993d93aaf03,data/org.eclipse.birt.data.tests/test/org/eclipse/birt/data/engine/olap/data/util/PrimarySortedStackTest.java::org.eclipse.birt.data.engine.olap.data.util.PrimarySortedStackTest::testNotDistinctDesc(),False,156,1,1,0,1,4,5,29,0,5,0,5,0,0,2,0,1,0,0,17,5,1,2,0,0,0,23,1,0,False,8857,8857,0,"0,0000",0,0,0,"0,0000","0,0000",0,"0,0000","0,0000",36,1,1.0,37.0,0.0,36.0,5.0,25.0,1.0,7.0,5.0,23.0,27.0,6.0,23.0,4.0,4.0,4.0,1.0,2.0,"0,03",4,4,4,1,29,37.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1797599,m2e,2025-09,55,bc9f5368675e410de0ad65a7883c70baf59e562c,org.eclipse.m2e.core/src/org/eclipse/m2e/core/internal/MvnProtocolHandlerService.java::org.eclipse.m2e.core.internal.MvnProtocolHandlerService::openConnection(URL),False,99,1,3,0,3,1,0,3,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,1,0,False,3241,2015,2,"0,5000",2019,324

In [18]:
from pathlib import Path

# Define o caminho de saída
caminho_saida = (
    Path(r"C:\Users\Kevin Strey\Desktop\Feature-models")
    / "7-DadosML"
    / "resultados"
    / "df_final.csv"
)

# Cria a pasta de destino, caso ainda não exista
caminho_saida.parent.mkdir(parents=True, exist_ok=True)

# Exporta o DataFrame para CSV
df_final.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV salvo em: {caminho_saida}")
print(f"Linhas exportadas: {len(df_final)}")
print(f"Colunas exportadas: {len(df_final.columns)}")

CSV salvo em: C:\Users\Kevin Strey\Desktop\Feature-models\7-DadosML\resultados\df_final.csv
Linhas exportadas: 1797604
Colunas exportadas: 75
